# cast

> Type-preserving casting utilities — the Julia analog of fasttransform's
> `cast.py`. Ported from the real AnswerDotAI/fasttransform source, not
> just the README (see `fasttransform_files/ARCHITECTURE.md` in the
> ReverseEngineered workspace for the full reverse-engineering notes).
>
> The one genuinely hard part to port faithfully: Python's version can
> *mutate* an existing object's type in place (`x.__class__ = typ`). A
> Julia value's concrete type is fixed forever once constructed — there is
> no equivalent. So `cast` here is always "construct a new value of the
> target type," never "mutate in place." This turns out to match exactly
> what Python's own `cast()` falls back to anyway when its in-place tricks
> don't apply, so the *user-visible* behavior — the result ends up being
> of the preserved/expected type — still matches.

In [ ]:
#| default_exp cast

In [ ]:
#| hide
using FTransform

## Metadata preservation

fasttransform lets a type opt in to carrying an arbitrary `_meta` payload
across a cast (used by fastai for things like remembering a tensor's
original filename). Python does this by duck-typing (`hasattr(res,
'set_meta')`); the Julia equivalent is an explicit trait: a type opts in
by defining its own `has_meta`/`get_meta`/`set_meta!` methods. This
collapses Python's two functions (`retain_meta` + `default_set_meta`)
into one coherent trait, since Julia has no monkey-patchable per-instance
attributes to distinguish "doesn't support meta" from "supports it but
hasn't set one yet" the way `hasattr` does.

In [ ]:
#| export
"""
    has_meta(x)

Does `x`'s type carry a `_meta`-style payload across a [`cast`](#cast)?
`false` by default — a type opts in by overriding this, [`get_meta`](#get_meta),
and [`set_meta!`](#set_meta!).
"""
has_meta(x) = false

"Metadata currently attached to `x` (only meaningful if `has_meta(x)`), or `nothing`."
get_meta(x) = nothing

"Attach `meta` to `x` in place, if `x`'s type supports it. No-op by default."
set_meta!(x, meta) = x

## `retain_meta`

In [ ]:
#| export
"""
    retain_meta(old, res)

Copy `old`'s metadata onto `res`, if `res`'s type supports it
([`has_meta`](#has_meta)) and doesn't already carry its own.
"""
function retain_meta(old, res)
    if has_meta(res) && get_meta(res) === nothing
        set_meta!(res, get_meta(old))
    end
    return res
end

## `cast`

A hook, `before_cast`, mirrors fasttransform's `typ._before_cast(x)` —
a type can pre-process the value before construction (e.g. unwrapping a
container). No-op by default.

In [ ]:
#| export
"Pre-process `x` before [`cast`](#cast) constructs a `T` from it. No-op by default, overridable per target type."
before_cast(x, ::Type) = x

"""
    cast(x, T)

Cast `x` to type `T` — in Julia this always means *constructing* a new
`T` from `x` (via `T`'s own constructor), never mutating `x` in place
(Python's version tries in-place mutation first; Julia's types can't be
mutated after construction, so there's nothing to try — this goes
straight to what Python falls back to anyway).
"""
function cast(x, T)
    res = T(before_cast(x, T))
    return retain_meta(x, res)
end

## `retain_type`

The core type-preservation policy `Transform` builds on. `ret_type`
follows fasttransform's own three-way split, expressed with Julia's own
`nothing`/`Nothing` distinction instead of Python's `Any`/`NoneType`:

- `ret_type` omitted (`nothing`, the default) — infer: if `old`'s value is
  already compatible with `new`'s own type, preserve `old`'s type;
  otherwise return `new` untouched.
- `ret_type` given as a real `Type` — cast to exactly that type.
- `ret_type` given as literally `Nothing` (the type) — disable preservation
  entirely, always return `new` exactly as produced.

In [ ]:
#| export
"""
    retain_type(new, old=nothing; ret_type::Union{Nothing,Type}=nothing)

Cast `new` to `ret_type` if given, or to `old`'s own type if `old`'s type
could plausibly have produced `new` (checked via `hasmethod(convert,
...)`, not `isa` — a Julia wrapper type like a custom `Real` subtype
is never an `isa` match for the plain numeric type it wraps the way a
real Python subclass would be, but it *can* `convert` to/from it, which
is the signal that actually carries over). No conversion if
`ret_type === Nothing` or `new === nothing`.
"""
function retain_type(new, old=nothing; ret_type::Union{Nothing,Type}=nothing)
    new === nothing && return new
    ret_type === Nothing && return new
    if ret_type === nothing
        old === nothing && return new
        rt = old isa Type ? old : typeof(old)
        hasmethod(convert, Tuple{Type{rt},typeof(new)}) || return new
        ret_type = rt
    end
    (ret_type === Nothing || isa(new, ret_type)) && return new
    return retain_meta(old, cast(new, ret_type))
end

## `retain_types` / `explode_types`

Recursive versions for tuples/vectors: cast each element of `new` to the
type of the matching element of `old`, pairing elements positionally.

In [ ]:
#| export
_is_listy(x) = x isa Union{AbstractVector,Tuple} && !(x isa AbstractString)

"""
    retain_types(new, old=nothing, typs=nothing)

Cast each element of `new` to the type of the matching element of `old`
(or of `typs`, if given directly), recursively, preserving `new`'s own
container kind (a `Tuple` stays a `Tuple`, matching Python's own
"reconstruct with new/old's type" final step — plain `map` over a
`Tuple`+`Vector` mix degrades to a `Vector`, silently losing that).
Falls back to plain [`retain_type`](#retain_type) for a non-listy `new`.
"""
function retain_types(new, old=nothing, typs=nothing)
    if !_is_listy(new)
        return retain_type(new, old; ret_type = typs)
    end
    oldv = old === nothing ? fill(nothing, length(new)) : old
    typsv = typs === nothing ? fill(nothing, length(new)) : typs
    mapped = [retain_types(n, o, t) for (n, o, t) in zip(new, oldv, typsv)]
    return new isa Tuple ? Tuple(mapped) : mapped
end

"""
    explode_types(o)

The type of `o`, or (recursively, for a listy `o`) a `Dict` mapping its
own type to the exploded types of its elements.
"""
function explode_types(o)
    _is_listy(o) || return typeof(o)
    return Dict(typeof(o) => [explode_types(x) for x in o])
end

## Tests

Julia can't literally subtype the built-in `Float64` the way Python
subtypes `float` (`Float64` is a `primitive type`, and only `abstract
type`s can have subtypes) — so the README's `FS <: float` example becomes
a wrapper struct here instead. The *behavior* being demonstrated (an
operation that would naturally degrade to a plain number instead
preserves the caller's wrapper type) is the same.

In [ ]:
#| hide
struct FS <: Real
    val::Float64
end
Base.Float64(x::FS) = x.val
FS(x::Real) = FS(Float64(x))
Base.:*(a::FS, b::Real) = Float64(a) * b
Base.:*(a::Real, b::FS) = a * Float64(b)

@assert retain_type(3.0) == 3.0
@assert retain_type(3.0, 2.0) == 3.0            # old is a float too -> ret_type is Float64, new already is -> untouched
f1 = FS(1.0)
val = f1 * 2.0                                   # ordinary arithmetic degrades to a plain Float64
@assert val isa Float64
preserved = retain_type(val, f1)
@assert preserved isa FS
@assert Float64(preserved) == 2.0
@assert retain_type(val, f1; ret_type=Nothing) isa Float64   # explicitly disabled -> no preservation
@assert retain_type(val, f1; ret_type=Float64) isa Float64   # explicit target type -> cast to exactly that
@assert retain_types((val, val), (f1, f1)) == (FS(2.0), FS(2.0))
@assert explode_types(5) == Int
@assert explode_types([1, 2]) == Dict(Vector{Int} => [Int, Int])